# Análise de Sarcopenia com Machine Learning

**Objetivo:** construir e avaliar um modelo experimental de classificação de sarcopenia utilizando informações de idosos, como idade, massa muscular e capacidade funcional.

> **Importante:** os dados são transversais. Este notebook estima a **classificação de sarcopenia no momento da avaliação**, e **não** prevê quem desenvolverá a condição nos próximos anos. Nenhum resultado constitui diagnóstico médico.

### Etapas do projeto

1. Importação das bibliotecas e dos dados
2. Exploração inicial e verificação de qualidade
3. Tratamento de valores numéricos e ausentes
4. Escolha da variável-alvo e das características
5. Divisão em treino e teste
6. Validação e comparação de classificadores
7. Métricas, matriz de confusão e interpretação
8. Limitações e próximos passos

## 1. Bibliotecas

Utilizamos **pandas** para manipular os dados, **matplotlib/seaborn** para gráficos e **scikit-learn** para construir o pipeline e avaliar modelos de **classificação**, que retornam categorias, como `0` e `1`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, recall_score, precision_score, roc_auc_score, RocCurveDisplay
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.inspection import permutation_importance

SEMENTE = 42

## 2. Fonte de dados

O conjunto original foi publicado por **Dequin e colaboradores**, vinculado a uma pesquisa da Universidade de Deusto (Bilbao). Ele reúne **1.253 observações** com informações antropométricas e de desempenho físico.

- [Repositório de origem](https://github.com/dmdequin/sarcopenia_and_machine_learning)
- [Artigo científico](https://doi.org/10.1371/journal.pone.0316174)

O CSV utiliza vírgula como separador decimal. Na leitura, erros de planilha como `#DIV/0!` e `#NULL!` são reconhecidos como ausentes, sem substituição por valores arbitrários.

In [ ]:
url = (
    "https://raw.githubusercontent.com/dmdequin/"
    "sarcopenia_and_machine_learning/master/data/datos_completos.csv"
)

df = pd.read_csv(
    url,
    decimal=",",
    na_values=["#NULL!", "#DIV/0!", ""],
    low_memory=False
)

df = df.rename(columns={
    "Edad": "Idade",
    "Grasa": "Gordura_Corporal",
    "Músculo": "Massa_Muscular",
    "Músculo relativ": "Musculo_Relativo",
    "Sarcopenia 1=SI;0=NO v/m": "Sarcopenia",
})

print("Dimensão da base:", df.shape)
display(df.head())

## 3. Qualidade dos dados e análise exploratória

Antes de ensinar o modelo, verificamos tipos, frequência dos diagnósticos e dados ausentes. Isso permite identificar problemas sem alterar o conjunto de dados original arbitrariamente.

In [ ]:
print("Tipos de dados:")
print(df.dtypes.to_string())

print("\nColunas com valores ausentes:")
faltantes = df.isna().sum()
display(faltantes[faltantes > 0].sort_values(ascending=False).to_frame("Quantidade ausente"))

print("\nDistribuição original da sarcopenia:")
print(df["Sarcopenia"].value_counts(dropna=False))

print("\nDescrição numérica:")
display(df.describe(include="number").T)

## 4. Preparação dos dados

A variável-alvo (`Sarcopenia`) vem codificada como `0 = não` e `1 = sim`. Mantemos apenas as linhas com esses rótulos válidos.

Para as **entradas**, selecionamos medidas clínicas/demográficas que podem ser informadas independentemente, como idade, peso, massa muscular e força. **Excluímos todas as outras colunas que já são classificações ou classificadores derivados do diagnóstico**, evitando que o modelo veja a resposta por outro nome.

**Ressalva metodológica:** algumas medidas, como massa muscular e velocidade de marcha, participam da própria definição clínica de sarcopenia. Seu uso pode fazer o modelo reaprender o critério utilizado para gerar o rótulo, não prever doença futura. Para uma avaliação de risco independente, seria necessária outra definição de desfecho e validação externa.

In [ ]:
# Campos numéricos efetivamente usados na modelagem
colunas_numericas = [
    "Idade", "Altura", "Peso", "Gordura_Corporal",
    "Massa_Muscular", "Musculo_Relativo",
    "Hand_Izq", "Hand_Drch", "Agil_m_s", "Sentadilla",
    "Mean Power", "Relative power"
]
colunas_categoricas = ["Sexo"]

# Conversão robusta: valores como "1,23" viram 1.23.
for coluna in colunas_numericas:
    valores = df[coluna].astype("string").str.strip().str.replace(",", ".", regex=False)
    df[coluna] = pd.to_numeric(valores, errors="coerce")

# Sexo é variável categórica: não deve ser tratado como escala numérica.
df["Sexo"] = df["Sexo"].astype("string")

df["Sarcopenia"] = pd.to_numeric(df["Sarcopenia"], errors="coerce")
df_modelo = df.loc[df["Sarcopenia"].isin([0, 1])].copy()
df_modelo["Sarcopenia"] = df_modelo["Sarcopenia"].astype(int)

variaveis = colunas_numericas + colunas_categoricas
X = df_modelo[variaveis].copy()
y = df_modelo["Sarcopenia"].copy()

print("Quantidade de registros válidos:", len(df_modelo))
print("Variáveis selecionadas:", X.columns.tolist())
print("Proporção das classes:\n", y.value_counts(normalize=True).sort_index().round(3))
display(X.head())

## 5. Treino e teste

Separamos **80% para treinamento** e **20% para teste**, com estratificação para preservar aproximadamente a mesma proporção de casos `0` e `1`. As medianas e demais parâmetros de pré-processamento só serão aprendidos dentro dos dados de treinamento por meio de um **Pipeline**, evitando vazamento estatístico.

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEMENTE
)

transformacao_numerica = Pipeline([
    ("imputacao", SimpleImputer(strategy="median")),
    ("escala", StandardScaler()),
])
transformacao_categorica = Pipeline([
    ("imputacao", SimpleImputer(strategy="most_frequent")),
    ("codificacao", OneHotEncoder(handle_unknown="ignore")),
])

preprocessamento = ColumnTransformer([
    ("numericas", transformacao_numerica, colunas_numericas),
    ("categoricas", transformacao_categorica, colunas_categoricas),
])

print("Treino:", X_treino.shape, "| Teste:", X_teste.shape)

## 6. Modelos de classificação

Agora usamos dois classificadores apropriados para uma saída `0` ou `1`:

- **Regressão Logística:** modelo linear de classificação, útil como linha de base interpretável.
- **Random Forest:** conjunto de árvores de decisão que aprende padrões não lineares.

Não utilizamos `LinearRegression`: ela prevê números contínuos (como `0.37` e `1.21`), incompatíveis com o cálculo direto do F1-score de classificação.

A comparação inicial usa validação cruzada estratificada **apenas dentro do conjunto de treinamento**. A avaliação do conjunto de teste permanece separada.

In [ ]:
algoritmos = {
    "Regressão Logística": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=SEMENTE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=SEMENTE,
        n_jobs=-1,
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
modelos = {}
resultados_cv = []

for nome, estimador in algoritmos.items():
    pipeline = Pipeline([
        ("preprocessamento", preprocessamento),
        ("classificador", estimador),
    ])
    scores = cross_validate(
        pipeline, X_treino, y_treino, cv=cv,
        scoring={"f1": "f1", "roc_auc": "roc_auc", "recall": "recall"},
        n_jobs=1,
        error_score="raise"
    )
    modelos[nome] = pipeline
    resultados_cv.append({
        "Modelo": nome,
        "F1 (CV)": scores["test_f1"].mean(),
        "Recall (CV)": scores["test_recall"].mean(),
        "ROC-AUC (CV)": scores["test_roc_auc"].mean(),
    })

comparacao_cv = pd.DataFrame(resultados_cv).sort_values("ROC-AUC (CV)", ascending=False)
display(comparacao_cv.round(4))

## 7. Avaliação final no conjunto de teste

Escolhemos o modelo pelo resultado da validação cruzada do **treino** e só então verificamos seu desempenho no **teste**. Analisamos:

- **Recall:** de todos os casos positivos, quantos o modelo identifica.
- **Precisão:** de todos os casos que o modelo indicou como positivos, quantos realmente eram positivos.
- **F1-score:** equilíbrio entre recall e precisão.
- **ROC-AUC:** capacidade de separar positivos e negativos em diferentes limiares.

Em estudos de triagem, é importante analisar os falsos negativos, mas nenhuma métrica isolada comprova utilidade clínica.

In [ ]:
melhor_nome = comparacao_cv.iloc[0]["Modelo"]
modelo_final = modelos[melhor_nome]
modelo_final.fit(X_treino, y_treino)

predicoes = modelo_final.predict(X_teste)
probabilidades = modelo_final.predict_proba(X_teste)[:, 1]

print("Modelo escolhido pelo ROC-AUC médio da validação cruzada:", melhor_nome)
print("F1-score (teste):", round(f1_score(y_teste, predicoes), 4))
print("Recall (teste):", round(recall_score(y_teste, predicoes), 4))
print("Precisão (teste):", round(precision_score(y_teste, predicoes, zero_division=0), 4))
print("ROC-AUC (teste):", round(roc_auc_score(y_teste, probabilidades), 4))
print("\nRelatório de classificação:\n")
print(classification_report(y_teste, predicoes, target_names=["Sem sarcopenia", "Com sarcopenia"], zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_teste, predicoes,
    display_labels=["Sem", "Com"],
    values_format="d", cmap="Blues"
)
plt.title("Matriz de confusão — conjunto de teste")
plt.show()

RocCurveDisplay.from_predictions(y_teste, probabilidades, name=melhor_nome)
plt.title("Curva ROC — conjunto de teste")
plt.show()

## 8. Quais variáveis mais influenciam as previsões?

A **importância por permutação** embaralha uma variável de cada vez e mede quanto o resultado piora. É uma medida **descritiva do modelo**, não prova de causa, nem um ranking clínico validado.

In [ ]:
importancias = permutation_importance(
    modelo_final, X_teste, y_teste,
    scoring="roc_auc", n_repeats=5,
    random_state=SEMENTE, n_jobs=1
)
ranking = pd.Series(importancias.importances_mean, index=X_teste.columns).sort_values()

display(ranking.sort_values(ascending=False).to_frame("Importância média (ROC-AUC)"))
plt.figure(figsize=(9, 6))
ranking.plot(kind="barh")
plt.xlabel("Queda média de ROC-AUC ao embaralhar a variável")
plt.title("Importância das variáveis — dados de teste")
plt.tight_layout()
plt.show()

## 9. Discussão, limitações e próximos passos

### O que os resultados realmente mostram?

O classificador foi desenvolvido com dados coletados no mesmo momento do diagnóstico e tenta reconhecer padrões associados a uma **classificação já atribuída**. Ele **não mede incidência futura** de sarcopenia.

### Limitações

- Base de um contexto populacional específico de idosos de Bilbao; resultados podem não se generalizar a outras populações.
- Possível **circularidade** entre variáveis clínicas usadas como entradas e critérios que produziram o rótulo.
- Desbalanceamento de classes e valores faltantes exigem avaliação cuidadosa.
- Valores de `predict_proba` **não são automaticamente riscos clínicos calibrados**.
- Estudo educacional/exploratório, sem validação externa.

### Próximas melhorias

1. Revisar o critério clínico do rótulo e comparar com um conjunto de entradas que não reproduza diretamente a regra diagnóstica.
2. Medir incerteza e calibração das probabilidades (ex.: Brier score e curva de calibração).
3. Repetir a avaliação em outra população e analisar resultados por sexo e faixa etária.
4. Usar uma coorte **longitudinal** com indivíduos inicialmente sem sarcopenia e desfecho registrado após alguns anos para o projeto de previsão futura.
5. Somente depois das validações considerar uma interface de apoio à triagem, sempre sem substituir avaliação profissional.

### Referência principal

Virto, N. et al. (2024). *Exploring determinant factors influencing muscle quality and sarcopenia in Bilbao's older adult population through machine learning: A comprehensive analysis approach.* PLOS ONE. https://doi.org/10.1371/journal.pone.0316174